# Delta Lake Incremental Data Processing — Superstore Dataset

**Objective:** Perform incremental data processing using Delta Lake — load data, clean it, simulate incremental data, apply MERGE (upsert), validate results, and display final output.

**Dataset:** Sample Superstore (retail orders). Primary key used for MERGE: **`Row ID`** (each row = one order line item).

- `superstore_master.csv` → orders from **2014–2016**
- `superstore_incremental.csv` → **new 2017 orders** (insert) + **15 existing 2016 orders with updated Discount/Profit/Ship Mode** (update) — simulates a real business scenario like a return/adjustment being recorded later plus new sales coming in.

**Environment:** Databricks Free Edition (serverless). Update the file paths in Step 1 to match wherever you uploaded the CSVs (e.g. `/Volumes/workspace/default/<folder>/...`).


## Step 0: Setup

In [ ]:
# Spark session is already available as `spark` on Databricks — no setup needed.
from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("Spark version:", spark.version)


## Step 1: Load Dataset into a Delta Table

📸 Screenshot for `screenshots/data_loading/`

In [ ]:
# UPDATE these two paths to match your uploaded volume paths
master_csv_path = "/Volumes/workspace/default/<your_folder>/superstore_master.csv"
delta_table_path = "/Volumes/workspace/default/<your_folder>/delta/superstore_orders"

df_master_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(master_csv_path)
)

print("Row count (raw master):", df_master_raw.count())
df_master_raw.printSchema()
df_master_raw.show(5, truncate=False)


In [ ]:
(
    df_master_raw.write
    .format("delta")
    .mode("overwrite")
    .save(delta_table_path)
)
print("Master data written to Delta table at:", delta_table_path)


## Step 2: Basic Cleaning (Handle Nulls, Remove Duplicates)

This Superstore dataset is already fairly clean, but we still run proper checks — a real pipeline should never assume data is clean.

📸 Screenshot for `screenshots/data_cleaning/`

In [ ]:
df_clean = spark.read.format("delta").load(delta_table_path)

print("Before cleaning:", df_clean.count(), "rows")

# Check nulls per column
df_clean.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_clean.columns]).show(vertical=True)

# Remove exact duplicate rows (if any)
df_clean = df_clean.dropDuplicates()

# Remove duplicate Row IDs, keeping the first occurrence (Row ID is our primary key)
df_clean = df_clean.dropDuplicates(["Row ID"])

# Drop rows missing the primary key or Order ID (can't process these)
df_clean = df_clean.filter(F.col("Row ID").isNotNull() & F.col("Order ID").isNotNull())

print("After cleaning:", df_clean.count(), "rows")


In [ ]:
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(delta_table_path)
)
print("Cleaned data saved back to Delta table.")


## Step 3: Create a Second Dataset (Simulated Incremental Data)

`superstore_incremental.csv` contains:
- **15 existing orders (Row IDs from the master set)** with updated `Discount`, `Profit`, and `Ship Mode` — simulating a correction/adjustment
- **All 2017 orders** — brand new order lines that don't exist in the master table yet

In [ ]:
incremental_csv_path = "/Volumes/workspace/default/<your_folder>/superstore_incremental.csv"

df_incremental = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(incremental_csv_path)
)

print("Incremental row count:", df_incremental.count())
df_incremental.show(5, truncate=False)


## Step 4: Apply MERGE (Upsert)

- If `Row ID` **already exists** in the Delta table → **UPDATE** that row with the incoming values
- If `Row ID` **does not exist** → **INSERT** it as a new row

📸 Screenshot for `screenshots/scd1/`

In [ ]:
delta_table = DeltaTable.forPath(spark, delta_table_path)

update_cols = [c for c in df_incremental.columns if c != "Row ID"]

(
    delta_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.`Row ID` = source.`Row ID`"
    )
    .whenMatchedUpdate(set={f"`{c}`": f"source.`{c}`" for c in update_cols})
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE completed.")


## Step 5: Validate Results (Row Count, Duplicates)

📸 Screenshot for `screenshots/validation/`

In [ ]:
df_final = spark.read.format("delta").load(delta_table_path)

before_count = df_clean.count()
after_count = df_final.count()
new_orders_count = df_incremental.join(df_clean, on="Row ID", how="left_anti").count()

print("Rows before MERGE:", before_count)
print("Rows after MERGE:", after_count)
print("Expected new rows inserted:", new_orders_count)
print("Row count check passed:", after_count == before_count + new_orders_count)

# Check for duplicate Row IDs (should be zero)
dupes = df_final.groupBy("Row ID").count().filter("count > 1")
print("Duplicate Row IDs found:", dupes.count())

# Spot-check a few of the updated Row IDs to confirm the update actually applied
updated_ids = [6334, 8934, 2070, 8868, 9080]
df_final.filter(F.col("Row ID").isin(updated_ids)).select(
    "Row ID", "Discount", "Profit", "Ship Mode"
).show()


## Step 6: Display Final Dataset and Summary

📸 Screenshot for `screenshots/final_output/`

In [ ]:
print("===== FINAL DELTA TABLE (sample) =====")
df_final.orderBy(F.col("Row ID")).show(10, truncate=False)

print("Total rows in final table:", df_final.count())

print("\n===== DELTA TABLE HISTORY (Time Travel Log) =====")
delta_table.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)


### Summary

- Loaded the Superstore **master dataset (2014–2016 orders)** into a Delta table.
- Ran cleaning checks for nulls and duplicate `Row ID`s (dataset was already clean, but the pipeline validates this rather than assuming it).
- Simulated an **incremental batch** containing 15 updated orders (Discount/Profit/Ship Mode corrections) plus all new 2017 orders.
- Used Delta Lake's **`MERGE INTO`** to upsert: matching `Row ID`s were updated, new `Row ID`s were inserted — all in a single atomic operation.
- Validated that the final row count equals `(cleaned master rows) + (brand-new rows)`, and confirmed there are **zero duplicate `Row ID`s**.
- Delta Lake's transaction log (`DESCRIBE HISTORY`) shows each write (initial load, cleaning overwrite, merge) as a separate, auditable version — enabling rollback/time-travel if something goes wrong.
